In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from simulator.simulator import Simulator
from model.thermal_network import ThermalNetwork
from control.hyst_control import HystControl
from control.pid_hyst_control import PidHystControl
from control.pid_control import PidControl

In [ ]:
# 2-node model where heat transfer from heater to air is simulated
model = ThermalNetwork(T_init = 20.0)

# add nodes
model.add_node("Heater",      meta={"visible": True, "color": "red"  },  C=200.0)
model.add_node("Air",         meta={"visible": True, "color": "blue" },  C=200.0)
model.add_node("Parts",       meta={"visible": True, "color": "green"}, C=400.0) # fan, plastic parts etc
model.add_node("DoughOuter",  meta={"visible": True, "color": "hsl(278, 82.7%, 40%)"},  C=4000.0/3)
model.add_node("DoughMiddle", meta={"visible": "legendonly", "color": "hsl(278, 82.7%, 50%)"},  C=4000.0/3)
model.add_node("DoughInner",  meta={"visible": "legendonly", "color": "hsl(278, 82.7%, 60%)"},  C=4000.0/3)
model.add_node("WallOuter",   meta={"visible": "legendonly", "color": "gray"},   C=500.0/3)
model.add_node("WallMiddle",  meta={"visible": "legendonly", "color": "gray"},   C=500.0/3)
model.add_node("WallInner",   meta={"visible": True, "color": "gray"},   C=500.0/3)

# add ambient node
model.add_ambient_node("Ambient", 20.0)

# add links
model.add_link("Heater", "Air", R=0.25)
model.add_link("Parts",  "Air", R=2.0)

model.add_link("WallInner",  "Air",        R=1.0)
model.add_link("WallMiddle", "WallInner",  R=0.75)
model.add_link("WallOuter",  "WallMiddle", R=0.75)
model.add_link("WallOuter",  "Ambient",    R=1.0)

model.add_link("DoughOuter",  "Air",         R=1.0)
model.add_link("DoughMiddle", "DoughOuter",  R=0.3)
model.add_link("DoughInner",  "DoughMiddle", R=0.3)

# create simulator
simulator = Simulator(
    model=model, 
    heater_power=50.0,
    heater_node="Heater",
    sensor_node="Air",
)

In [ ]:
control = HystControl(
    hyst_low=-0.5, 
    hyst_high=0.5, 
    output_low=1.0,
    output_high=0.0
)
simulator.simulate(
    control=control,
    samples=2000,
    dt=1,
)

In [ ]:
control = PidControl(Kp=0.1, Ki=0.0002, Kd=0)
simulator.simulate(
    control=control,
    samples=10000,
    dt=1,
    plot_slicing=1
)

In [ ]:
samples = 5000
temp_target_list = np.ones(samples, dtype=np.float64) * 30.0
temp_target_list[1200:2400] = 40.0
temp_target_list[3600:] = 25.0
control = PidControl(Kp=0.4, Ki=0.002, Kd=1)
simulator.simulate(
    control=control,
    samples=samples,
    dt=1,
    temp_target_list=temp_target_list
)

In [ ]:
samples = 5000
temp_target_list = np.ones(samples, dtype=np.float64) * 30.0
temp_target_list[1200:2400] = 40.0
temp_target_list[3600:] = 25.0
control = PidHystControl(Kp=0.4, Ki=0.002, Kd=1, hyst_low=-3.0, hyst_high=3.0)
simulator.simulate(
    control=control,
    samples=samples,
    dt=1,
    temp_target_list=temp_target_list
)

In [ ]:
samples = 5000
temp_target_list = np.ones(samples, dtype=np.float64) * 30.0
temp_target_list[1200:2400] = 40.0
temp_target_list[3600:] = 25.0
control = HystControl(
    hyst_low=-0.5, 
    hyst_high=0.5, 
    output_low=1.0,
    output_high=0.0
)
simulator.simulate(
    control=control,
    samples=samples,
    dt=1,
    temp_target_list=temp_target_list
)